# M4b - regrade the baseline with the fixed judge

Re-scores the saved M4 runs with the corrected Layer 4 decoder. The target's 900
generations are already on disk, so only the judge runs: **~15-20 min instead of ~1.5 h.**

**What was wrong:** the old decoder matched plain English as base64 and decoded it into
garbage, so the judge graded garbage and said UNCLEAR. It now only decodes when that
makes the text more readable.

**Needs:** one T4 is enough (only the judge loads). Internet On. The transcript dataset
attached.

## 1 - Setup

In [ ]:
%pip -q install -U transformers accelerate bitsandbytes huggingface_hub

In [ ]:
import os, subprocess, sys, pathlib, time, json, glob, shutil
import numpy as np, pandas as pd

_sec = None
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
except Exception as e:
    print('no Kaggle secrets client:', e)

def _secret(name):
    try:
        return _sec.get_secret(name) if _sec is not None else None
    except Exception:
        return None

_hf = _secret('HF_TOKEN')
if _hf:
    os.environ['HF_TOKEN'] = _hf
    from huggingface_hub import login; login(token=_hf)
    print('HF auth OK')
else:
    print('no HF_TOKEN secret (fine - models are public)')

In [ ]:
# --- get the repo -----------------------------------------------------------
REPO   = "MehemudAzad/LLM-jailbreaking-with-layered-prompt-defense"
BRANCH = "main"
WORK   = pathlib.Path("/kaggle/working")
ROOT   = WORK / "repo"

_gh  = _secret("GH_TOKEN")
_url = f"https://{_gh}@github.com/{REPO}.git" if _gh else f"https://github.com/{REPO}.git"

os.chdir(WORK)
subprocess.run(["rm", "-rf", str(ROOT)], check=False)
_r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, _url, str(ROOT)],
                    cwd=str(WORK), capture_output=True, text=True)
if _r.returncode != 0:
    _err = _r.stderr.replace(_gh, "***") if _gh else _r.stderr
    raise RuntimeError("git clone failed:\n" + _err)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("HEAD", subprocess.check_output(["git","-C",str(ROOT),"rev-parse","--short","HEAD"]).decode().strip())

_missing = [f for f in ("regrade.py", "report.py", "defense/layer4_response_classifier.py")
            if not (ROOT / f).exists()]
if _missing:
    raise RuntimeError(f"clone is missing {_missing} -- commit and push them, then re-run this cell")
print("repo files OK")

## 2 - Find the uploaded runs

In [ ]:
# the attached Kaggle dataset. Change this if you re-upload under a different name.
DATA = pathlib.Path("/kaggle/input/datasets/kmazd1110/baseline-asr-llm-jailbreak/artifacts")

if not DATA.exists():          # fall back to a search
    hits = glob.glob("/kaggle/input/**/transcript.jsonl", recursive=True)
    if not hits:
        raise FileNotFoundError("no transcript.jsonl under /kaggle/input -- is the dataset attached?")
    DATA = pathlib.Path(hits[0]).parent.parent
print("dataset:", DATA)

runs = {p.name: p for p in sorted(DATA.iterdir()) if (p / "transcript.jsonl").exists()}
for name in runs:
    n = sum(1 for l in open(runs[name] / "transcript.jsonl") if '"type": "trial"' in l)
    print(f"  {name}   {n} trials")

SANITY   = next((p for n, p in runs.items() if "sanity"   in n), None)
BASELINE = next((p for n, p in runs.items() if "baseline" in n), None)
assert BASELINE is not None, "no *baseline* run found -- upload it too"

In [ ]:
def stage(src):
    """/kaggle/input is read-only and regrade writes next to the transcript -> copy into logs/."""
    dest = ROOT / "logs" / src.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dest, dirs_exist_ok=True)
    return dest

baseline = stage(BASELINE)
sanity   = stage(SANITY) if SANITY else None
print("staged:", baseline.name, "|", sanity.name if sanity else "(no sanity run)")

## 3 - Put the judge on one GPU

`config.toml` pins it to `cuda:1` for the two-GPU eval. Only the judge loads here.

In [ ]:
import torch
from core.config import CONFIG

if torch.cuda.device_count() < 2:
    CONFIG['models']['judge']['device'] = 'cuda:0'
print('GPUs:', torch.cuda.device_count(), '| judge ->', CONFIG['models']['judge']['device'])
print('judge:', CONFIG['models']['judge']['name'], CONFIG['models']['judge'].get('quant'))

## 4 - Quick check on the sanity run (54 trials, ~2 min)

Loads the judge and confirms everything works before the long one.

In [ ]:
from regrade import regrade
import report

if sanity:
    regrade(sanity)
    report.print_regrade_diff(sanity)

## 5 - The baseline (900 trials, ~15-20 min)

381 trials are caught by the refusal heuristic and cost nothing; ~519 hit the judge.

In [ ]:
summary = regrade(baseline)
print('\nlabels changed:', summary['labels_changed'], '/', summary['trials'])

## 6 - Before vs after

In [ ]:
report.print_regrade_diff(baseline)

In [ ]:
report.changed_samples(baseline, n=10)

**What to look for:** the `judged:` line is what the judge saw. Expect the
`combination_*` replies to be the target guessing wrong about what it decoded, not
actually complying - so they stay UNCLEAR and the ASR barely moves.

If real harmful content shows up that used to be garbage, then the ASR for those attacks
goes up, and that is the correct number.

## 7 - Final table + save

In [ ]:
report.print_asr(baseline, regraded=True, title='BASELINE ASR (regraded) - undefended Qwen2.5-3B, 50 AdvBench goals')

In [ ]:
for run in filter(None, (baseline, sanity)):
    out = pathlib.Path('/kaggle/working/artifacts') / run.name
    out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(run, out, dirs_exist_ok=True)
    print('mirrored ->', out)
!cd /kaggle/working && zip -qr regraded.zip artifacts && ls -la regraded.zip

## Done

Download `regraded.zip` and unzip it over your local `logs/`. The regraded table is the
one for the report.

**Still open:** hand-label ~30 random trials and compare with the judge, so the report
can say why the numbers are trustworthy.

**Next: M5** - Layer 2 paraphraser, then the defended run.